# Jak komputer czyta tekst - od liczenia słów do wektorów

Ten notatnik to companion do blog posta: [Jak komputer czyta tekst](jak-komputer-czyta-tekst.html)

Uruchamiaj komórki po kolei, czytaj wyjaśnienia i eksperymentuj!

**Instalacja:**
```
pip install tiktoken transformers tokenizers scikit-learn pandas gensim numpy matplotlib seaborn
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")
print("Wszystkie biblioteki załadowane!")

---
## 1. Tokenizacja - jak komputer tnie tekst na kawałki

Komputer nie widzi liter. Nie widzi słów. Komputer widzi **liczby**.

Pierwszy krok to **tokenizacja** - podział tekstu na małe kawałki (tokeny), które da się zamienić na liczby.

Zobaczmy, jak różne modele dzielą to samo polskie zdanie.

### Tiktoken - tokenizatory GPT-2 i GPT-4o

`tiktoken` to biblioteka od OpenAI. `gpt2` to stary tokenizer (słaby z polskim), `cl100k_base` to nowszy (lepszy).

Zwróć uwagę na `encode` (tekst → liczby) i `decode` (liczby → tekst).

In [ ]:
import tiktoken

text = "Nieprawdopodobnie szczęśliwy"

gpt2 = tiktoken.get_encoding("gpt2")
gpt4 = tiktoken.get_encoding("cl100k_base")

print("=== Encode (tekst → token IDs) ===")
print(f"GPT-2: {gpt2.encode(text)}")
print(f"GPT-4o: {gpt4.encode(text)}")

print("\n=== Decode (token IDs → kawałki tekstu) ===")
print(f"GPT-2: {[gpt2.decode([t]) for t in gpt2.encode(text)]}")
print(f"GPT-4o: {[gpt4.decode([t]) for t in gpt4.encode(text)]}")

print(f"\nGPT-2: {len(gpt2.encode(text))} tokenów")
print(f"GPT-4o: {len(gpt4.encode(text))} tokenów")

Widzisz? GPT-2 tnie polski tekst na mnóstwo małych kawałków, bo nie "zna" polskiego. GPT-4o radzi sobie lepiej.

A jak to wygląda z modelami trenowanymi specjalnie na polskim?

### Polskie tokenizatory: HerBERT i Polish RoBERTa

- **HerBERT** - model Allegro, trenowany na polskim tekście z internetu
- **Polish RoBERTa** - model trenowany na ~20 GB polskiego tekstu

Obydwa "znają" polski znacznie lepiej niż GPT. Zobaczmy:

In [ ]:
from transformers import AutoTokenizer

text = "Nieprawdopodobnie szczęśliwy"

herbert = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
roberta = AutoTokenizer.from_pretrained("sdadas/polish-roberta-base-v2")

print(f'HerBERT:  {herbert.tokenize(text)}  ({len(herbert.tokenize(text))} tokenów)')
print(f'RoBERTa:  {roberta.tokenize(text)}  ({len(roberta.tokenize(text))} tokenów)')

Trzy tokeny zamiast kilkunastu! Bo polskie tokenizatory mają polskie słowa w słowniku.

Zwizualizujmy to porównanie dla kilku zdań:

In [ ]:
test_texts = [
    "Nieprawdopodobnie szczęśliwy",
    "Kot siedzi na macie i śpi",
    "Szczęście to stan umysłu",
]

results = {}
for text in test_texts:
    results[text] = {
        "GPT-2": len(gpt2.encode(text)),
        "GPT-4o": len(gpt4.encode(text)),
        "HerBERT": len(herbert.tokenize(text)),
        "Polish RoBERTa": len(roberta.tokenize(text)),
    }

df_tokens = pd.DataFrame(results).T
display(df_tokens)

fig, ax = plt.subplots(figsize=(10, 4))
df_tokens.plot(kind="barh", ax=ax, colormap="Set2")
ax.set_xlabel("Liczba tokenów")
ax.set_title("Porównanie tokenizatorów na polskim tekście")
ax.legend(title="Tokenizer")
for container in ax.containers:
    ax.bar_label(container, fontsize=8)
plt.tight_layout()
plt.show()

GPT-2 potrzebuje najwięcej tokenów na polski tekst, a polskie tokenizatory (HerBERT, Polish RoBERTa) najmniej. To dlatego były trenowane specjalnie na polskim korpusie.

**Eksperyment:** Zmień `test_texts` na własne zdania i zobacz jak się zachowują!

### Własny tokenizer BPE - budujemy od zera

A gdybyśmy chcieli stworzyć **własny** tokenizator na własnym korpusie? Biblioteka `tokenizers` od HuggingFace pozwala to zrobić w kilku linijkach.

- `vocab_size=300` - maksymalna liczba tokenów w słowniku
- `BPE` - algorytm Byte Pair Encoding (ten sam co w GPT)
- `Whitespace` - najpierw dzielimy po spacjach

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=300, special_tokens=["[UNK]"])

corpus = [
    "Ala ma kota i psa",
    "Kot siedzi na macie",
    "Pies biega po parku",
    "Ala ma kota i kota i jeszcze raz kota",
    "Kot lubi mleko i kot lubi spać",
    "Pies lubi biegać po parku i gonić kota",
    "Szczęśliwy kot to kot który ma dużo mleka",
    "Nieprawdopodobnie szczęśliwy pies biega po parku",
    "Szczęśliwy Ala ma kota i psa i szczęśliwy dom",
    "Dom to miejsce gdzie mieszka szczęśliwa rodzina",
]

tokenizer.train_from_iterator(corpus, trainer)
print(f"Rozmiar słownika: {tokenizer.get_vocab_size()}")

test = "Nieprawdopodobnie szczęśliwy"
encoding = tokenizer.encode(test)
print(f'"{test}" → {encoding.tokens}')

test2 = "szczęśliwy dom"
encoding2 = tokenizer.encode(test2)
print(f'"{test2}" → {encoding2.tokens}')

Dwa tokeny! Bo nasz mały korpus jest po polsku i "Nieprawdopodobnie" oraz "szczęśliwy" wystąpiły często.

**Eksperyment:** Zmień `vocab_size` na 50 i zobacz co się stanie - tokenizer będzie musiał ciąć na mniejsze kawałki!

---
## 2. Od tokenów do liczb: BoW i TF-IDF

Mamy tokeny. Ale jak je **policzyć**? Najprostszy sposób: **Bag of Words (BoW)** - zliczamy ile razy każde słowo występuje w dokumencie.

Traktujemy dokument jak "torebkę słów" - bez kolejności, bez gramatyki, tylko zliczenia.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "Ten film jest świetny",
    "Ten film jest beznadziejny",
    "Świetny film polecam",
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

df_bow = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)
print("Bag of Words - zliczenia słów:")
display(df_bow)

Zauważ: "film" jest wszędzie (1 w każdym wierszu). "beznadziejny" tylko w Dok 2. Ale BoW traktuje "film" tak samo jak "beznadziejny" - a przecież "beznadziejny" mówi nam więcej o dokumencie!

Rozwiązanie: **TF-IDF** - waży słowa nie tylko po tym, jak często są w jednym dokumencie, ale też po tym, jak **rzadkie** są w całym korpusie.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)
print("TF-IDF - ważność słów w dokumentach:")
display(df_tfidf.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

sns.heatmap(df_bow, annot=True, cmap="Blues", ax=axes[0], cbar_kws={"label": "Liczba"})
axes[0].set_title("Bag of Words (zliczenia)")

sns.heatmap(df_tfidf, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[1], cbar_kws={"label": "TF-IDF"})
axes[1].set_title("TF-IDF (ważność)")

plt.tight_layout()
plt.show()

Porównaj: "film" ma wartość 1 wszędzie w BoW, ale niskie TF-IDF - bo jest w każdym dokumencie i nie wnosi informacji. "beznadziejny" ma wysokie TF-IDF w Dok 2 - bo to jedyne miejsce gdzie występuje.

### TF-IDF jako wyszukiwarka

TF-IDF ma praktyczne zastosowanie: **przeszukiwanie tekstu**. Zamieniamy zapytanie na wektor TF-IDF i szukamy dokumentu najbardziej do niego podobnego (podobieństwo kosinusowe).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query = "świetny film"
query_vec = tfidf_vectorizer.transform([query])
similarities = cosine_similarity(query_vec, tfidf_matrix)[0]

ranked = sorted(zip(corpus, similarities), key=lambda x: -x[1])
print(f'Zapytanie: "{query}"\n')
for i, (doc, sim) in enumerate(ranked, 1):
    marker = " <-- najlepszy wynik!" if i == 1 else ""
    print(f'{i}. "{doc}" → podobieństwo: {sim:.3f}{marker}')

fig, ax = plt.subplots(figsize=(8, 3))
colors = ["#2ecc71" if s == max(similarities) else "#bdc3c7" for s in similarities]
ax.barh([f"Dok {i+1}" for i in range(len(corpus))], similarities, color=colors)
ax.set_xlabel("Podobieństwo kosinusowe")
ax.set_title(f'Wyniki wyszukiwania dla: "{query}"')
for i, s in enumerate(similarities):
    ax.text(s + 0.01, i, f'{s:.3f}', va='center')
plt.tight_layout()
plt.show()

**Eksperyment:** Zmień `query` na "beznadziejny" albo "polecam" i zobacz jak zmienią się wyniki!

### N-gramy - dodajemy kontekst

BoW i TF-IDF ignorują **kolejność** słów. "nie lubię" i "lubię" to dla nich dwa osobne słowa, a nie przeciwieństwo.

**N-gramy** rozwiązują ten problem: zamiast liczyć pojedyncze słowa, liczymy **pary** (bigramy), **trójki** (trigramy) itd.

W sklearn wystarczy zmienić jeden parametr: `ngram_range=(1, 2)` oznacza "unigramy + bigramy".

In [ ]:
vectorizer_ngram = TfidfVectorizer(ngram_range=(1, 2))
tfidf_ngram = vectorizer_ngram.fit_transform(corpus)

df_ngram = pd.DataFrame(
    tfidf_ngram.toarray(),
    columns=vectorizer_ngram.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)

fig, ax = plt.subplots(figsize=(14, 3))
sns.heatmap(df_ngram, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "TF-IDF"})
ax.set_title("N-gramy (1,2) z TF-IDF - unigramy + bigramy")
plt.tight_layout()
plt.show()

print(f"Unigramy: {len(TfidfVectorizer().fit(corpus).get_feature_names_out())} cech")
print(f"Unigramy + bigramy: {len(vectorizer_ngram.get_feature_names_out())} cech")

Słownik cech urósł! Teraz mamy zarówno pojedyncze słowa ("film", "jest") jak i pary ("film jest", "jest świetny", "jest beznadziejny").

---
## 3. Łańcuchy Markowa - pierwszy "generator tekstu"

BoW i TF-IDF potrafią **analizować** tekst, ale nie potrafią **generować** nowego.

Łańcuchy Markowa to najprostszy model, który potrafi generować tekst. Idea: dla każdego słowa patrzymy, jakie słowa najczęściej po nim następują, i wybieramy następne probabilistycznie.

Zbudujemy to na małym korpusie:

In [ ]:
import random
from collections import defaultdict, Counter

corpus_text = "ala ma kota ala ma psa kot lubi mleko ala lubi kota"
tokens = corpus_text.split()
print(f"Korpus: {corpus_text}")
print(f"Słów: {len(tokens)}, unikalnych: {len(set(tokens))}")

Budujemy słownik trigramów - każda trójka słów mapuje na możliwe następne słowo:

In [ ]:
trigrams = {}
for i in range(len(tokens) - 3):
    key = (tokens[i], tokens[i + 1], tokens[i + 2])
    next_word = tokens[i + 3]
    if key not in trigrams:
        trigrams[key] = []
    trigrams[key].append(next_word)

print("Słownik trigramów:")
for key, values in trigrams.items():
    print(f'  {" ".join(key)} → {values}')

Teraz generujemy! Zaczynamy od trigramu startowego (to jest nasz "prompt"):

In [ ]:
random.seed(42)
current = ("ala", "ma", "kota")  # to jest nasz "prompt"!
output = list(current)

for _ in range(10):
    if tuple(output[-3:]) not in trigrams:
        break
    possibilities = trigrams[tuple(output[-3:])]
    output.append(random.choice(possibilities))

print("Wygenerowany tekst:")
print(" ".join(output))

Na tym małym korpusie wynik jest przewidywalny (każdy trigram ma tylko jedną opcję). Ale zobaczmy **macierz przejść** - prawdopodobieństwa, że po słowie A nastąpi słowo B:

In [ ]:
transitions = defaultdict(Counter)
for i in range(len(tokens) - 1):
    transitions[tokens[i]][tokens[i + 1]] += 1

all_words = sorted(set(tokens))
transition_matrix = np.zeros((len(all_words), len(all_words)))
word2i = {w: i for i, w in enumerate(all_words)}

for word, nexts in transitions.items():
    total = sum(nexts.values())
    for n, count in nexts.items():
        transition_matrix[word2i[word]][word2i[n]] = count / total

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    transition_matrix,
    xticklabels=all_words,
    yticklabels=all_words,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    ax=ax,
    cbar_kws={"label": "P(przejście)"},
)
ax.set_xlabel("Następne słowo")
ax.set_ylabel("Obecne słowo")
ax.set_title("Macierz przejść - łańcuch Markowa (bigramy)")
plt.tight_layout()
plt.show()

Wiersz = obecne słowo, kolumna = następne słowo. "ala" → "ma" ma 67%, "ala" → "lubi" ma 33%.

**Ciekawostka:** Łańcuchy Markowa to pierwowzór dzisiejszych LLM! ChatGPT też przewiduje następne słowo, tylko używa Transformerów zamiast prostej macierzy przejścia.

---
## 4. Naive Bayes - klasyfikator spamu

Łańcuchy Markowa generują tekst. A co jeśli chcemy **klasyfikować** tekst? Np. rozpoznawać spam.

Naive Bayes mówi: "każde słowo głosuje na jedną z kategorii". Jeśli w mailu jest "viagra" - głosuje na spam. Jeśli "raport" - głosuje na nie-spam.

- `CountVectorizer` - zamienia tekst na wektor zliczeń słów (BoW)
- `MultinomialNB(alpha=1.0)` - klasyfikator Naive Bayes z Laplace smoothing (alpha=1.0)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

emails = [
    "Kup viagra tanio teraz",
    "Viagra darmowa oferta",
    "Spotkanie jutro o 10",
    "Prześlij mi raport jutro",
    "Tanie leki online kup teraz",
    "Raport z wczoraj prześlij",
]
labels = [1, 1, 0, 0, 1, 0]  # 1=spam, 0=nie-spam

bow = CountVectorizer()
X = bow.fit_transform(emails)

classifier = MultinomialNB(alpha=1.0)
classifier.fit(X, labels)

new_email = "Viagra jutro spotkanie"
new_X = bow.transform([new_email])
prediction = classifier.predict(new_X)
probs = classifier.predict_proba(new_X)[0]

print(f'Mail: "{new_email}"')
print(f'Klasyfikacja: {"SPAM" if prediction[0] == 1 else "NIE-SPAM"}')
print(f'P(spam) = {probs[1]:.1%}, P(nie-spam) = {probs[0]:.1%}')

Zobaczmy, które słowa "głosują" na spam, a które na nie-spam:

In [ ]:
feature_names = bow.get_feature_names_out()
log_probs_spam = classifier.feature_log_prob_[1]
log_probs_ham = classifier.feature_log_prob_[0]

df_probs = pd.DataFrame({
    "słowo": feature_names,
    "P(słowo|spam)": np.exp(log_probs_spam),
    "P(słowo|nie-spam)": np.exp(log_probs_ham),
}).sort_values("P(słowo|spam)", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(df_probs))
width = 0.35
ax.bar([i - width/2 for i in x], df_probs["P(słowo|spam)"], width, label="Spam", color="#e74c3c", alpha=0.8)
ax.bar([i + width/2 for i in x], df_probs["P(słowo|nie-spam)"], width, label="Nie-spam", color="#2ecc71", alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(df_probs["słowo"], rotation=45, ha="right")
ax.set_ylabel("Prawdopodobieństwo")
ax.set_title("P(słowo | klasa) - które słowa wskazują na spam?")
ax.legend()
plt.tight_layout()
plt.show()

"viagra" i "kup" mają wysokie P(słowo|spam), a "raport" i "prześlij" mają wysokie P(słowo|nie-spam). To są właśnie te "głosy"!

---
## 5. Word2Vec - słowa stają się wektorami

BoW, TF-IDF, Naive Bayes - wszystkie te metody traktują słowa jako **atomy**, bez znaczenia. "Kot" i "pies" to dwa zupełnie różne słowa, choć oba to zwierzęta.

**Word2Vec** zmienia wszystko: każde słowo zamienia w wektor (np. 300 liczb) w taki sposób, że **podobne słowa mają podobne wektory**.

Zbudujemy to **od zera**, w czystym numpy!

### Krok 1: Słownik i one-hot encoding

Każde słowo zaczyna jako **one-hot wektor** - zera wszędzie i jedna jedynka na pozycji tego słowa.

In [ ]:
np.random.seed(42)

corpus_w2v = "kot siedzi na macie i śpi pies siedzi na dywanie i śpi kot biega po pokoju pies biega po parku".split()
vocab = list(set(corpus_w2v))
word2idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)

def one_hot(word):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1
    return vec

print(f'Słownik ({vocab_size} słów): {sorted(vocab)}')
print(f'\nOne-hot "kot": {one_hot("kot")}')
print(f'One-hot "pies": {one_hot("pies")}')
print(f'\nProblem: kot i pies są tak samo daleko jak kot i "po"!')

### Krok 2: Macierz wag i pary treningowe

Tworzymy dwie macierze wag:
- **W1** (słownik × wymiar embeddingu) - to będą nasze embeddingi po treningu
- **W2** (wymiar embeddingu × słownik) - do przewidywania słowa

Następnie tworzymy pary treningowe (CBOW): **kontekst → słowo środkowe**.

In [ ]:
embed_dim = 5  # w prawdziwym Word2Vec to 100-300
W1 = np.random.randn(vocab_size, embed_dim) * 0.01
W2 = np.random.randn(embed_dim, vocab_size) * 0.01

window = 2
pairs = []
for i in range(window, len(corpus_w2v) - window):
    context = corpus_w2v[i - window:i] + corpus_w2v[i + 1:i + window + 1]
    target = corpus_w2v[i]
    pairs.append((context, target))

print(f'Liczba par treningowych: {len(pairs)}')
print(f'\nPrzykładowe pary (kontekst → cel):')
for ctx, tgt in pairs[:5]:
    print(f'  {ctx} → "{tgt}"')

### Krok 3: Trening (gradient descent)

Dla każdej pary:
1. **Forward:** uśredniamy wektory kontekstu i przewidujemy środek
2. **Loss:** cross-entropy (jak bardzo się pomyliliśmy)
3. **Backward:** aktualizujemy wagi W1 i W2

Po treningu **wiersz W1 odpowiadający danemu słowu to jego embedding**.

In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

losses = []
lr = 0.05

for epoch in range(500):
    loss = 0
    for context_words, target_word in pairs:
        context_idx = [word2idx[w] for w in context_words]
        target_idx = word2idx[target_word]

        hidden = np.mean(W1[context_idx], axis=0)
        output = softmax(hidden @ W2)
        loss -= np.log(output[target_idx] + 1e-8)

        grad = output.copy()
        grad[target_idx] -= 1
        W2 -= lr * np.outer(hidden, grad)
        grad_hidden = grad @ W2.T
        for idx in context_idx:
            W1[idx] -= lr * grad_hidden / len(context_idx)

    losses.append(loss)
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}/500, loss: {loss:.3f}')

print(f'\nTrening zakończony! Loss spadł z {losses[0]:.1f} do {losses[-1]:.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color="#3498db", linewidth=1)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (cross-entropy)")
ax.set_title("Jak spada loss w trakcie treningu CBOW")
ax.set_xlim(0, len(losses))
plt.tight_layout()
plt.show()

### Krok 4: Sprawdzamy wynik - embeddingi!

Po treningu każdy wiersz macierzy W1 to wektor (embedding) danego słowa. Zobaczmy czy podobne słowa mają podobne wektory:

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print("Embeddingi wybranych słów:")
for word in ["kot", "pies", "siedzi", "biega", "parku"]:
    print(f'  {word}: {W1[word2idx[word]].round(3)}')

print("\nPodobieństwa kosinusowe:")
pairs_sim = [("kot", "pies"), ("kot", "siedzi"), ("kot", "biega"), ("kot", "parku"), ("pies", "biega")]
for w1, w2 in pairs_sim:
    sim = cosine(W1[word2idx[w1]], W1[word2idx[w2]])
    print(f'  {w1} ↔ {w2}: {sim:.3f}')

**kot ↔ pies** powinno mieć najwyższe podobieństwo - bo oba to zwierzęta występujące w podobnym kontekście. Sieć sama to odkryła, bez żadnych etykiet!

### Wizualizacja: Word2Vec z gensim + PCA

Nasz model od zera działa, ale na małym korpusie. Użyjmy teraz `gensim` - gotowej biblioteki Word2Vec - na trochę większym korpusie, żeby zobaczyć embeddingi w 2D (za pomocą PCA).

In [ ]:
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

w2v_corpus = [
    ["kot", "siedzi", "na", "macie", "i", "śpi"],
    ["pies", "siedzi", "na", "dywanie", "i", "śpi"],
    ["kot", "biega", "po", "pokoju"],
    ["pies", "biega", "po", "parku"],
    ["kot", "i", "pies", "bawią", "się", "w", "ogrodzie"],
    ["kot", "je", "karmę", "z", "miski"],
    ["pies", "je", "karmę", "z", "miski"],
    ["kot", "łapie", "mysz", "w", "domu"],
    ["pies", "goni", "kota", "po", "ogrodzie"],
    ["ptak", "siedzi", "na", "gałęzi", "drzewa"],
    ["ryba", "pływa", "w", "akwarium"],
    ["samochód", "jeździ", "po", "drodze"],
    ["rower", "jeździ", "po", "ścieżce"],
    ["kot", "mruczy", "gdy", "głaszczesz", "go"],
    ["pies", "merda", "ogonem", "gdy", "widzi", "pana"],
]

model = Word2Vec(
    sentences=w2v_corpus,
    vector_size=10,
    window=3,
    min_count=1,
    sg=0,       # 0 = CBOW, 1 = Skip-gram
    epochs=200,
)

print(f'Słownik: {len(model.wv)} słów')
print(f'\nNajbardziej podobne do "kot":')
for word, score in model.wv.most_similar("kot", topn=5):
    print(f'  {word}: {score:.3f}')

In [ ]:
words = list(model.wv.key_to_index.keys())
vectors = np.array([model.wv[w] for w in words])

pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(coords[:, 0], coords[:, 1], s=80, alpha=0.7, edgecolors="black", linewidth=0.5)
for i, word in enumerate(words):
    ax.annotate(word, (coords[i, 0] + 0.01, coords[i, 1] + 0.01), fontsize=9)
ax.set_title(f"Word2Vec embeddingi (PCA, variance: {pca.explained_variance_ratio_.sum():.0%})")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
plt.tight_layout()
plt.show()

Na tym małym korpusie klastry nie są wyraźne. Ale na dużym korpusie (polska Wikipedia) "kot" i "pies" byłyby blisko, "samochód" i "rower" blisko, a "kot" i "samochód" daleko.

### Heatmapa podobieństw między słowami

In [ ]:
selected = ["kot", "pies", "ptak", "ryba", "samochód", "rower", "siedzi", "biega", "je", "śpi"]
selected = [w for w in selected if w in model.wv]

sim_matrix = np.zeros((len(selected), len(selected)))
for i, w1 in enumerate(selected):
    for j, w2 in enumerate(selected):
        sim_matrix[i][j] = model.wv.similarity(w1, w2)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    sim_matrix,
    xticklabels=selected,
    yticklabels=selected,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    ax=ax,
    cbar_kws={"label": "Podobieństwo kosinusowe"},
)
ax.set_title("Podobieństwo kosinusowe między słowami")
plt.tight_layout()
plt.show()

Najwyższe wartości powinny być na przekątnej (słowo vs. siebie = 1.00). Poza nią: kot↔pies powinno być wyższe niż kot↔samochód.

### Bonus: Word2Vec + łańcuchy Markowa

Word2Vec sam nie generuje tekstu. Ale jeśli połączymy go z łańcuchami Markowa - tak żeby słowa bardziej podobne do kontekstu miały większą szansę być wybrane - dostajemy lepszy generator!

In [ ]:
from collections import defaultdict

corpus_gen = [
    "kot siedzi na macie i śpi",
    "pies siedzi na dywanie i śpi",
    "kot biega po pokoju",
    "pies biega po parku",
    "kot i pies bawią się w ogrodzie",
    "kot je karmę z miski",
    "pies je karmę z miski",
    "kot łapie mysz w domu",
    "pies goni kota po ogrodzie",
    "ptak siedzi na gałęzi drzewa",
    "kot patrzy przez okno i mruczy",
    "pies szczeka na listonosza",
    "kot śpi cały dzień na kanapie",
]

tokenized = [sentence.split() for sentence in corpus_gen]
w2v_model = Word2Vec(sentences=tokenized, vector_size=10, window=3, min_count=1, epochs=200)

transitions = defaultdict(list)
for sentence in tokenized:
    for i in range(len(sentence) - 1):
        transitions[sentence[i]].append(sentence[i + 1])

def generate(start, length=6):
    words = [start]
    for _ in range(length):
        current = words[-1]
        if current not in transitions:
            break

        candidates = transitions[current]
        context_vector = np.mean(
            [w2v_model.wv[w] for w in words[-3:] if w in w2v_model.wv],
            axis=0,
        )

        scored = []
        for candidate in candidates:
            if candidate in w2v_model.wv:
                similarity = np.dot(context_vector, w2v_model.wv[candidate]) / (
                    np.linalg.norm(context_vector) * np.linalg.norm(w2v_model.wv[candidate]) + 1e-8
                )
                scored.append((candidate, similarity))

        if scored:
            weights = [score + 1 for _, score in scored]
            chosen = random.choices([w for w, _ in scored], weights=weights, k=1)[0]
            words.append(chosen)
        else:
            words.append(random.choice(candidates))
    return " ".join(words)

print("=== Wygenerowane teksty (start: kot) ===")
for _ in range(5):
    print(f"  {generate('kot')}")

---
## Podsumowanie

W tym notatniku przeszliśmy całą drogę od tekstu do wektorów:

1. **Tokenizacja** - tekst → kawałki (tokeny)
2. **BoW / TF-IDF** - kawałki → liczby (zliczenia i ważność)
3. **N-gramy / Łańcuchy Markowa** - dodajemy kontekst i generujemy tekst
4. **Naive Bayes** - klasyfikujemy tekst na podstawie prawdopodobieństw
5. **Word2Vec** - słowa stają się wektorami znaczeniowymi

To są fundamenty, na których zbudowane są dzisiejsze LLM (GPT, Claude, Gemini).

**Co dalej?** W następnym wpisie wejdziemy w Transformer-y i mechanizm attention - gdzie kontekst zmienia znaczenie każdego słowa.